In [7]:
import pandas as pd
import numpy as np
import re

print("=" * 60)
print("        SEARCH QUERY SPELLING CORRECTOR")
print("=" * 60)

df = pd.read_csv("Spelling_Error_Dataset.csv")

print("\nDATASET INFORMATION")
print("-" * 60)
print("Number of rows    :", len(df))
print("Number of columns :", len(df.columns))

print("\nCOLUMN NAMES")
print("-" * 60)

for column in df.columns:
    print("-", column)

print("\nFIRST 5 RECORDS")
print("-" * 60)
print(df.head().to_string(index=False))

print("\nMISSING VALUES")
print("-" * 60)
print(df.isnull().sum())

df = df.dropna()

df["correct_word"] = df["correct_word"].astype(str).str.lower()
df["misspelled_word"] = df["misspelled_word"].astype(str).str.lower()

vocabulary = set(df["correct_word"])

spelling_dictionary = dict(
    zip(df["misspelled_word"], df["correct_word"])
)

print("\nVOCABULARY INFORMATION")
print("-" * 60)
print("Correct words :", len(vocabulary))
print("Error pairs   :", len(spelling_dictionary))


def edit_distance(word1, word2):

    rows = len(word1) + 1
    columns = len(word2) + 1

    matrix = np.zeros((rows, columns), dtype=int)

    for i in range(rows):
        matrix[i][0] = i

    for j in range(columns):
        matrix[0][j] = j

    for i in range(1, rows):

        for j in range(1, columns):

            if word1[i - 1] == word2[j - 1]:
                cost = 0
            else:
                cost = 1

            matrix[i][j] = min(
                matrix[i - 1][j] + 1,
                matrix[i][j - 1] + 1,
                matrix[i - 1][j - 1] + cost
            )

    return matrix[rows - 1][columns - 1]


def correct_word(word):

    word = word.lower()

    if word in vocabulary:
        return word

    if word in spelling_dictionary:
        return spelling_dictionary[word]

    candidates = []

    for candidate in vocabulary:

        if abs(len(word) - len(candidate)) <= 2:

            distance = edit_distance(word, candidate)

            candidates.append((distance, candidate))

    if candidates:

        candidates.sort(key=lambda x: (x[0], x[1]))

        return candidates[0][1]

    return word


def correct_query(query):

    words = query.lower().split()

    corrected_words = []
    incorrect_words = []
    suggestions = []

    for word in words:

        clean_word = re.sub(r"[^a-zA-Z]", "", word)

        if clean_word == "":
            continue

        if clean_word in vocabulary:

            corrected_words.append(clean_word)

        else:

            corrected = correct_word(clean_word)

            incorrect_words.append(clean_word)

            suggestions.append((clean_word, corrected))

            corrected_words.append(corrected)

    final_query = " ".join(corrected_words)

    return incorrect_words, suggestions, final_query


print("\n" + "=" * 60)
print("TESTING SPELLING CORRECTOR")
print("=" * 60)

test_queries = [
    "machne lerning cours",
    "computr scince studnt",
    "artifical inteligence",
    "programing languge",
    "softwear developmant"
]

for number, query in enumerate(test_queries, 1):

    incorrect_words, suggestions, final_query = correct_query(query)

    print("\nTEST", number)
    print("-" * 60)

    print("Original Query:")
    print(query)

    print("\nIncorrect Words:")

    for word in incorrect_words:
        print("-", word)

    print("\nSuggested Corrections:")

    for wrong, correct in suggestions:
        print(wrong, "->", correct)

    print("\nCorrected Query:")
    print(final_query)


print("\n" + "=" * 60)
print("USER SEARCH QUERY")
print("=" * 60)

query = input("Enter your search query: ")

incorrect_words, suggestions, final_query = correct_query(query)

print("\nOriginal Query:")
print(query)

print("\nIncorrect Words:")

if incorrect_words:

    for word in incorrect_words:
        print("-", word)

else:

    print("No spelling errors found.")

print("\nSuggested Corrections:")

if suggestions:

    for wrong, correct in suggestions:
        print(wrong, "->", correct)

else:

    print("No corrections required.")

print("\nFinal Corrected Query:")
print(final_query)

print("\n" + "=" * 60)
print("        SPELLING CORRECTION COMPLETED")
print("=" * 60)

        SEARCH QUERY SPELLING CORRECTOR

DATASET INFORMATION
------------------------------------------------------------
Number of rows    : 500
Number of columns : 2

COLUMN NAMES
------------------------------------------------------------
- correct_word
- misspelled_word

FIRST 5 RECORDS
------------------------------------------------------------
correct_word misspelled_word
     student        studennt
     teacher        teacheer
      school          scholo
     college         cellego
  university       uniersity

MISSING VALUES
------------------------------------------------------------
correct_word       0
misspelled_word    0
dtype: int64

VOCABULARY INFORMATION
------------------------------------------------------------
Correct words : 403
Error pairs   : 500

TESTING SPELLING CORRECTOR

TEST 1
------------------------------------------------------------
Original Query:
machne lerning cours

Incorrect Words:
- machne
- lerning
- cours

Suggested Corrections:
machne -> ma

Enter your search query:  scince



Original Query:
scince

Incorrect Words:
- scince

Suggested Corrections:
scince -> science

Final Corrected Query:
science

        SPELLING CORRECTION COMPLETED
